# langchain 프롬프트 템플릿 실습

In [1]:
# api key 설정
from dotenv import load_dotenv
import os

load_dotenv(override=True)
api_key = os.environ["OPENAI_API_KEY"]

## 텍스트 format 출력

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from rich.console import Console
from rich.markdown import Markdown

console = Console()

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 프롬프트 (모든 내용을 system 메시지에 통합)
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 {place} 베테랑 여행 가이드입니다.
            고객 최적의 {place} {travel} 일정 수립을 도와줍니다."""),
    ("human", "해당 장소 {place}의 {travel} 일정에 맞는 여행 계획을 제안해 주세요.")
])


# 출력 파서 정의
output_parser = StrOutputParser()


# 체인 구성
chain = prompt | llm | output_parser

In [3]:
# 실행
input_data = {
    "place": "서울",
    "travel": "3일"
}
result = chain.invoke(input_data)

In [4]:
# 예쁘게 출력
md = Markdown(result)
console.print(md)

서울에서의 3일 여행 일정은 다양한 문화, 역사, 음식, 쇼핑을 경험할 수 있도록 구성해 보았습니다. 아래의 일정을 참고해
주세요.                                                                                                            

                                              1일차: 역사와 문화 탐방                                              

오전                                                                                                               

 • 경복궁: 조선 왕조의 대표적인 궁궐로, 아름다운 건축과 정원을 감상할 수 있습니다. 국립민속박물관도 함께 방문해    
   보세요.                                                                                                         
 • 청와대: 경복궁 인근에 위치한 청와대의 외부를 둘러보며 한국의 대통령 관저를 확인해 보세요.                       

점심                                                                                                               

 • 인사동: 전통 한국 음식점에서 점심 식사를 하며, 인사동 거리의 전통 공예품 가게를 구경하세요.                     

오후                                                                                                               

 • 북촌 한옥마을: 전통 한옥이 잘 보존된 지역으로, 산책하며 고즈넉한 분위기를 즐길 수 있습니다.                     
 • 창덕궁: 유네스코 세계문화유산으로 지정된 궁궐로, 후원 투어를 추천합니다.                                        

저녁                                                                                                               

 • 종로: 전통 한식당에서 저녁 식사 후, 근처의 맛집이나 카페를 찾아보세요.                                          

                                              2일차: 현대 서울과 쇼핑                                              

오전                                                                                                               

 • 남산서울타워(N 서울타워): 케이블카를 타고 올라가 서울의 전경을 감상하세요.                                      
 • 남산공원: 타워 주변의 아름다운 공원에서 산책하며 자연을 느껴보세요.                                             

점심                                                                                                               

 • 명동: 명동 거리의 다양한 길거리 음식을 즐기세요. (떡볶이, 크레페 등)                                            

오후                                                                                                               

 • 명동 쇼핑: 다양한 브랜드 매점과 화장품 가게에서 쇼핑을 즐기세요.                                                
 • 동대문 디자인 플라자(DDP): 현대적인 건축물과 디자인 관련 전시를 구경할 수 있습니다.                             

저녁                                                                                                               

 • 이태원: 다국적 음식점이 많은 이태원에서 저녁을 즐기세요. (태국 음식, 이탈리안 등)                               

                                             3일차: 자연과 여유의 시간                                             

오전                                                                                                               

 • 한강 공원: 자전거를 대여해 한강을 따라 산책하거나 자전거 타기를 즐기세요.                                       
 • 여의도: 여의도 공원에서 여유로운 시간을 보내세요.                                                               

점심                                                                                                               

 • 여의도: 근처의 맛집에서 점심을 드세요.                                                                          

오후                                                                                                               

 • 홍대: 홍익대학교 주변의 예술적이고 자유로운 분위기를 즐기며, 다양한 상점과 카페를 탐방하세요.                   
 • 합정: 다양한 카페와 소품 가게가 있는 합정 동네를 즐겨보세요.                                                    

저녁                                                                                                               

 • 강남: 강남의 유명한 음식점에서 저녁을 즐기고, 강남역 주변의 쇼핑몰이나 거리에서 마무리하세요.                   

이 일정을 바탕으로 서울에서의 멋진 시간을 보내시길 바랍니다! 여행 중 궁금한 점이 있으면 언제든지 물어보세요.

## JsonOutputParser 사용

In [5]:
# 출력을 위한 JSON 스키마 정의
json_schema = """
{{
  "destination": "목적지",
  "duration": "기간",
  "overview": "여행 개요",
  "daily_plans": [
   {{
      "day": 1,
      "title": "제목",
      "morning": "오전 일정",
      "lunch": "점심 추천",
      "afternoon": "오후 일정",
      "dinner": "저녁 추천",
      "accommodation": "숙소 추천"
    }}
  ],
  "tips": ["팁1", "팁2"],
  "total_budget": "예상 총 비용"
}}
"""

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from rich.console import Console
from rich.panel import Panel
import json

console = Console()

# LLM 초기화
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0.5,
    model_kwargs={"response_format": {"type": "json_object"}}
    )

# 프롬프트 (모든 내용을 system 메시지에 통합)
prompt = ChatPromptTemplate.from_messages([
    ("system", f"""당신은 {{place}} 베테랑 여행 가이드입니다.
            고객 최적의 {{place}} {{travel}} 일정 수립을 도와줍니다.
    
     다음의 JSON 형식으로 응답하세요.
     {json_schema}"""),
    ("human", "해당 장소 {place}의 {travel} 일정에 맞는 여행 계획을 여행 계획을 제안해주세요.")
])

# Json 출력 파서 정의
output_parser = JsonOutputParser()


# 체인 구성
chain = prompt | llm | output_parser

In [7]:
# llm 호출
input_data = {
    "place": "서울",
    "travel": "3일"
}
result = chain.invoke(input_data)

In [8]:
json_data = json.dumps(result, indent=2, ensure_ascii=False)
console.print(json_data)

{
  "destination": "서울",
  "duration": "3일",
  "overview": "서울의 전통과 현대가 어우러진 매력적인 도시를 탐험하는 3일간의 여행입니다. 고궁, 맛집, 쇼핑, 그리고 
야경까지 다양한 경험을 제공합니다.",
  "daily_plans": [
    {
      "day": 1,
      "title": "서울의 전통과 역사 탐방",
      "morning": "경복궁 방문 및 국립민속박물관 관람",
      "lunch": "광화문 인근의 한정식 전문점에서 점심",
      "afternoon": "북촌 한옥마을 산책 및 인사동 거리 탐방",
      "dinner": "인사동의 전통 찻집에서 저녁",
      "accommodation": "종로 지역의 전통 한옥 게스트하우스"
    },
    {
      "day": 2,
      "title": "현대 서울의 매력",
      "morning": "남산 서울타워 방문 및 케이블카 체험",
      "lunch": "명동의 유명한 닭갈비 식당",
      "afternoon": "명동 쇼핑 및 서울 시청 광장 탐방",
      "dinner": "홍대의 이자카야에서 저녁",
      "accommodation": "홍대 근처의 부티크 호텔"
    },
    {
      "day": 3,
      "title": "서울의 자연과 야경",
      "morning": "한강공원 자전거 타기 또는 산책",
      "lunch": "한강공원 내의 피크닉 도시락",
      "afternoon": "63빌딩 전망대 방문 및 여의도 공원 산책",
      "dinner": "여의도에서 한정판 스시 저녁",
      "accommodation": "여의도 지역의 고급 호텔"
    }
  ],
  "tips": [
    "대중교통을 이용하면 편리합니다. T-money 카드 구매 추천.",
    "서울의 야경은 정말 아름답습니다. 꼭 즐겨보세요."
  ],
  "total_budget": "약 60만원"
}

In [9]:
# 예쁘게 출력
# 성공 - 결과 출력
console.print(Panel(
    json.dumps(result, indent=2, ensure_ascii=False),
    title="[bold green]여행 계획 (JSON)[/bold green]",
    border_style="green"
))

╭─────────────────────────────────────────────── 여행 계획 (JSON) ────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "destination": "서울",                                                                                        │
│   "duration": "3일",                                                                                            │
│   "overview": "서울의 전통과 현대가 어우러진 매력적인 도시를 탐험하는 3일간의 여행입니다. 고궁, 맛집, 쇼핑,     │
│ 그리고 야경까지 다양한 경험을 제공합니다.",                                                                     │
│   "daily_plans": [                                                                                              │
│     {                                                                                                           │
│       "day": 1,                                                                                                 │
│       "title": "서울의 전통과 역사 탐방",                                                                       │
│       "morning": "경복궁 방문 및 국립민속박물관 관람",                                                          │
│       "lunch": "광화문 인근의 한정식 전문점에서 점심",                                                          │
│       "afternoon": "북촌 한옥마을 산책 및 인사동 거리 탐방",                                                    │
│       "dinner": "인사동의 전통 찻집에서 저녁",                                                                  │
│       "accommodation": "종로 지역의 전통 한옥 게스트하우스"                                                     │
│     },                                                                                                          │
│     {                                                                                                           │
│       "day": 2,                                                                                                 │
│       "title": "현대 서울의 매력",                                                                              │
│       "morning": "남산 서울타워 방문 및 케이블카 체험",                                                         │
│       "lunch": "명동의 유명한 닭갈비 식당",                                                                     │
│       "afternoon": "명동 쇼핑 및 서울 시청 광장 탐방",                                                          │
│       "dinner": "홍대의 이자카야에서 저녁",                                                                     │
│       "accommodation": "홍대 근처의 부티크 호텔"                                                                │
│     },                                                                                                          │
│     {                                                                                                           │
│       "day": 3,                                                                                                 │
│       "title": "서울의 자연과 야경",                                                                            │
│       "morning": "한강공원 자전거 타기 또는 산책",                                                              │
│       "lunch": "한강공원 내의 피크닉 도시락",                                                                   │
│       "afternoon": "63빌딩 전망대 방문 및 여의도 공원 산책",                                                    │
│       "dinner": "여의도에서 한정판 스시 저녁",                                                                  │
│       "accommodation": "여의도 지역의 고급 호텔"                                                                │
│     }                                                                                                           │
│   ],                                                                                                            │
│   "tips": [                                                                                                     │
│     "대중교통을 이용하면 편리합니다. T-money 카드 구매 추천.",                                                  │
│     "서울의 야